<a href="https://colab.research.google.com/github/tribhuwan-kandpal/sdxl-model/blob/sdxl_lcm_resolution/sdxl_lcm_diffusers_colab_resolution_profiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q diffusers transformers accelerate peft

from diffusers import UNet2DConditionModel, DiffusionPipeline, LCMScheduler
import torch

unet = UNet2DConditionModel.from_pretrained("latent-consistency/lcm-sdxl", torch_dtype=torch.float16, variant="fp16")
pipe = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", unet=unet, torch_dtype=torch.float16, variant="fp16")

pipe.scheduler = LCMScheduler.from_config(pipe.scheduler.config)
pipe.to("cuda")

In [ ]:
import os
import torch
import numpy as np
from timeit import default_timer as timer
from google.colab import drive

# Mount Google Drive with error handling
try:
    drive.mount('/content/drive')
except Exception as e:
    print("Drive is already mounted or there was an error:", str(e))
    print("Continuing with the script...")

# Create directory if it doesn't exist with error handling
save_path = "/content/drive/MyDrive/sdxl"  # Changed path to use MyDrive
try:
    os.makedirs(save_path, exist_ok=True)
except OSError as e:
    print(f"Error creating directory: {e}")
    print("Attempting to proceed anyway...")

# Change working directory with error handling
try:
    os.chdir(save_path)
    print("Successfully changed to directory:", os.getcwd())
except OSError as e:
    print(f"Error changing directory: {e}")
    print("Will continue with current directory...")

prompt = "a picture of a tiger"
seed = 128
generator = torch.Generator("cuda").manual_seed(seed)

n_runs = 5
resolutions = [(1024, 1024), (768, 768)]

for res in resolutions:
    times = []
    print(f"\nTesting resolution: {res[0]}x{res[1]}")

    for i in range(n_runs):
        start = timer()

        image = pipe(
            prompt,
            height=res[0],
            width=res[1],
            num_inference_steps=4,
            guidance_scale=8.0,
            generator=generator
        ).images[0]

        end = timer()
        times.append(end - start)
        print(f"Run {i+1}: {times[-1]:.2f} seconds")

        # Save the image with error handling
        try:
            image_path = os.path.join(save_path, f"tiger_{res[0]}x{res[1]}_run_{i+1}.png")
            image.save(image_path)
            print(f"Saved image to: {image_path}")
        except Exception as e:
            print(f"Error saving image: {e}")


    print(f"Average generation time: {np.mean(times):.2f} seconds")
    print(f"Standard deviation: {np.std(times):.2f} seconds")

print("\nScript completed!")
